# OONCA: agente de apoio ao operador on-chain

Protótipo de análise de uma operação hipotética antes da assinatura. Usa `first_order_df.csv` e `wallet_metadata.csv`. O score é uma regra demonstrativa, não uma probabilidade de fraude. `isError` significa erro histórico. O agente não assina nem envia transações.

Esta versão inclui:
- **Features Comportamentais & Proxies de Automação** (`app/features/wallet_behavior.py`): cadência inter-transação ($\Delta t$), *rapid-fire ratio*, índice de *burstiness*, entropia horária normalizada e concentração HHI de destinatários.
- **Decomposição Determinística & Recomendações Contrafactuais** (`app/models/risk.py`): detalhamento por regra (`rule_breakdown`), lacunas de dados (`missing_data_flags`) e condições contrafactuais para reduzir o risco.
- **Trilha de Auditoria e Responsabilização** (`app/audit/trail.py`): hash SHA-256 canônico da evidência (`evidence_sha256`), flag de revisão humana obrigatória e índice de completude de governança.
- **Painel Visual de 4 Quadrantes & Simulação em Lote** (`app/visualization/risk.py`, `batch_analyze`).


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.agent.graph import analyze, batch_analyze
from app.ingestion.schema import ProposedOperation
from app.models.risk import RiskPolicy
from app.visualization.risk import plot_assessment, plot_operator_dashboard


Edite origem, destino e valor antes da execução. Com `use_llm=False`, a análise não chama o OpenRouter; mude para `True` para gerar uma explicação em linguagem natural. `as_of` opcional restringe o histórico a transações anteriores à data informada.


In [ ]:
operation = ProposedOperation.from_values(
    wallet_from='0xc2d7cf95645d33006175b78989035c7c9061d3f9',
    wallet_to='0x002f0c8119c16d310342d869ca8bf6ace34d9c39',
    amount='0.5',
    operation='transfer',
    asset='ETH',
    operator_id='mesa_tesouraria_01',
)
result = analyze(operation, policy=RiskPolicy(), use_llm=False)
print(result['explanation'])
print('Histórico:', result['profile']['n_rows'], 'linhas;', result['profile']['n_unique_tx_hashes'], 'hashes únicos')
print('Metadados Origem:', result['profile']['wallet_metadata'])
print('Metadados Destino:', result['profile']['recipient_metadata'])
print('Proxy de Automação:', result['profile']['automation_proxy'])
print('Contrafactuais:', result['assessment']['counterfactuals'])
print('Auditoria SHA-256:', result['audit_record']['evidence_sha256'])
plot_assessment(result)


## Simulação Comparativa em Lote (`batch_analyze`)

Avalia múltiplos cenários operacionais hipotéticos (ex.: transferência recorrente vs. novo destinatário vs. valor extremo acima do teto da política) e consolida decisões, scores heurísticos, proxies de automação e hashes de auditoria em um `DataFrame`.


In [ ]:
from decimal import Decimal

scenarios = [
    ProposedOperation.from_values(
        wallet_from='0xc2d7cf95645d33006175b78989035c7c9061d3f9',
        wallet_to='0x002f0c8119c16d310342d869ca8bf6ace34d9c39',
        amount='0.5',
        operator_id='mesa_tesouraria_01',
    ),
    ProposedOperation.from_values(
        wallet_from='0xc2d7cf95645d33006175b78989035c7c9061d3f9',
        wallet_to='0x9999999999999999999999999999999999999999',
        amount='25.0',
        operator_id='mesa_tesouraria_01',
    ),
    ProposedOperation.from_values(
        wallet_from='0xc2d7cf95645d33006175b78989035c7c9061d3f9',
        wallet_to='0x002f0c8119c16d310342d869ca8bf6ace34d9c39',
        amount='150.0',
        operator_id='mesa_tesouraria_01',
    ),
]

df_batch = batch_analyze(
    scenarios,
    policy=RiskPolicy(max_amount_eth=Decimal('100.0')),
    use_llm=False,
)
df_batch[['wallet_to', 'amount', 'decision', 'risk_score', 'recipient_seen', 'automation_flagged', 'human_review_required', 'accountability_score', 'reasons']]


## Limitações atuais

A amostra contém transações de 2016–2019 e pode repetir `TxHash` em várias linhas. O agente reporta contagem de linhas e hashes separadamente. As carteiras sem metadados não recebem um rótulo inventado. ENS e tipo de carteira foram consultados no presente, então podem não representar o estado na data das transações. Proxies temporais (`rapid_fire_ratio`, `burstiness_index`, `hourly_entropy_normalized`) sinalizam apenas padrões compatíveis com execução automatizada/scriptada e **não provam autoria por IA**. Sem dados de token, calldata, gas, preço, rótulo de fraude ou operador autenticado on-chain, a decisão é uma triagem preliminar. Um `APPROVE` indica somente que nenhuma regra deste protótipo pediu revisão; não autoriza uma transação real.
